# Emotion Classification with Bidirectional LSTM

Classifies text into 6 emotions: **anger, joy, sadness, fear, love, surprise**

**Model architecture:**
- Embedding layer
- Bidirectional LSTM
- Dropout (regularization)
- Dense output (softmax)

**Fixes vs original notebook:**
- Validation split added → EarlyStopping now works correctly
- Model + tokenizer saved to disk
- Paths are relative (no hardcoded local paths)
- Dropout added to reduce overfitting
- Confidence scores included in output

In [ ]:
import os
import pickle
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Bidirectional, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

print(f'TensorFlow version: {tf.__version__}')

In [ ]:
# ── Configuration — edit these paths ─────────────────────────────────────────
TRAIN_PATH = 'data/train.txt'   # semicolon-delimited: text;label
TEST_PATH  = 'data/Data.csv'    # CSV with a 'Speech' column
OUTPUT_DIR = 'models'

# Hyperparameters
MAX_SEQUENCE_LENGTH = 100
EMBEDDING_DIM       = 100
LSTM_UNITS          = 64
DENSE_UNITS         = 32
DROPOUT_RATE        = 0.3
EPOCHS              = 10
BATCH_SIZE          = 32
VALIDATION_SPLIT    = 0.1   # 10% of training data used for validation
PATIENCE            = 3     # EarlyStopping patience

# Label mapping
LABEL_TO_ID = {'anger': 0, 'joy': 1, 'sadness': 2, 'fear': 3, 'love': 4, 'surprise': 5}
ID_TO_LABEL = {v: k for k, v in LABEL_TO_ID.items()}

os.makedirs(OUTPUT_DIR, exist_ok=True)

## 1. Load Data

In [ ]:
# Load training data (text;label format)
with open(TRAIN_PATH, 'r', encoding='utf-8') as f:
    lines = f.readlines()

data = [line.strip().split(';') for line in lines if line.strip()]
train_speeches, train_labels_raw = zip(*data)

print(f'Training samples : {len(train_speeches)}')
print(f'Label distribution: {pd.Series(train_labels_raw).value_counts().to_dict()}')

In [ ]:
# Load test data
test_data = pd.read_csv(TEST_PATH)
test_speeches = test_data['Speech'].tolist()

print(f'Test samples: {len(test_speeches)}')
test_data.head()

## 2. Preprocess

In [ ]:
# Encode labels → integers → one-hot
train_labels_numeric = [LABEL_TO_ID[label] for label in train_labels_raw]
train_labels_onehot  = tf.keras.utils.to_categorical(
    train_labels_numeric, num_classes=len(LABEL_TO_ID)
)

In [ ]:
# Tokenize — fit on training text only
tokenizer = Tokenizer()
tokenizer.fit_on_texts(train_speeches)

vocab_size = len(tokenizer.word_index) + 1
print(f'Vocabulary size: {vocab_size}')

X_train = pad_sequences(
    tokenizer.texts_to_sequences(train_speeches), maxlen=MAX_SEQUENCE_LENGTH
)
X_test = pad_sequences(
    tokenizer.texts_to_sequences(test_speeches), maxlen=MAX_SEQUENCE_LENGTH
)

print(f'X_train shape: {X_train.shape}')
print(f'X_test  shape: {X_test.shape}')

## 3. Build Model

In [ ]:
model = Sequential([
    Embedding(vocab_size, EMBEDDING_DIM, input_length=MAX_SEQUENCE_LENGTH),
    Bidirectional(LSTM(LSTM_UNITS)),
    Dropout(DROPOUT_RATE),          # regularization — not in original
    Dense(DENSE_UNITS, activation='relu'),
    Dropout(DROPOUT_RATE),
    Dense(len(LABEL_TO_ID), activation='softmax'),
])

model.compile(
    loss='categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

model.summary()

## 4. Train

In [ ]:
callbacks = [
    # FIX: validation_split > 0 means val_loss is now available
    EarlyStopping(monitor='val_loss', patience=PATIENCE,
                  restore_best_weights=True, verbose=1),
    ModelCheckpoint(filepath=os.path.join(OUTPUT_DIR, 'best_model.keras'),
                    monitor='val_loss', save_best_only=True, verbose=1),
]

history = model.fit(
    X_train, train_labels_onehot,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_split=VALIDATION_SPLIT,   # FIX: was missing in original
    callbacks=callbacks,
)

## 5. Evaluate & Save

In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(history.history['loss'],     label='Train Loss')
ax1.plot(history.history['val_loss'], label='Val Loss')
ax1.set_title('Loss'); ax1.legend()

ax2.plot(history.history['accuracy'],     label='Train Acc')
ax2.plot(history.history['val_accuracy'], label='Val Acc')
ax2.set_title('Accuracy'); ax2.legend()

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'training_curves.png'), dpi=150)
plt.show()

In [ ]:
# Save tokenizer so predict.py can reuse it
with open(os.path.join(OUTPUT_DIR, 'tokenizer.pkl'), 'wb') as f:
    pickle.dump(tokenizer, f)
print('Tokenizer saved.')

## 6. Predict on Test Set

In [ ]:
preds = model.predict(X_test)
predicted_labels      = [ID_TO_LABEL[np.argmax(p)] for p in preds]
predicted_confidences = [float(np.max(p)) for p in preds]

test_data['Predicted Emotion'] = predicted_labels
test_data['Confidence']        = predicted_confidences

out_path = os.path.join(OUTPUT_DIR, 'predictions.csv')
test_data.to_csv(out_path, index=False)

print(f'Predictions saved to {out_path}')
test_data[['Speech', 'Predicted Emotion', 'Confidence']].head(10)